# 04. CIFAR-100, up to thirty deletions (M4)

This is the run that goes past the limit.

Notebook 03 measured that each deletion uses up 4 of the model's 64 available directions. Divide
and you get 16. After sixteen deletions there is nothing left to work with, so the seventeenth
request has no room to do anything.

The paper reports results at twenty and thirty deletions. This notebook deletes thirty classes
one at a time and looks at what actually happened to each one.

It also runs the prediction that makes the limit a real claim rather than a description: change
the size of each patch to 2, 4 and 8, and the limit should move to 32, 16 and 8 to match. If it
lands anywhere else, our understanding of the method is wrong.

Runs on CPU. Expect roughly fifteen minutes, most of it training 360 small patches.

## Words used in this notebook

Same vocabulary as notebook 03, with three additions at the end. If you have read that one, skip
to **Budget**.

- **Head.** The model's last layer, 64 numbers in and 100 class scores out. The only part we
  touch.
- **Adapter.** A small trainable correction to the head, one per deletion, built from 4
  independent directions.
- **Rank.** How many independent directions something uses. Our adapters use 4.
- **Projection.** The rule that stops a new adapter from reusing directions an earlier one
  claimed.
- **Free rank.** How many of the 64 directions nobody has claimed yet. Starts at 64, falls by 4
  per deletion.
- **Lambda.** How hard to apply the deletions. Higher forgets harder and damages more.
- **Retain accuracy.** Accuracy on classes we kept. Higher is better.
- **Forget accuracy.** Accuracy on classes we deleted. Lower is better.

New here:

- **Budget.** 64 divided by the rank, so 16 deletions at rank 4. The point where free rank hits
  zero and no further adapter can be built.
- **Checkpoint.** We train one sequence of 30 deletions and read the results after 5, 10, 20 and
  30 of them. That is sound because deletions happen in order: the first 10 of a 30-deletion run
  are the same 10 a 10-deletion run would have trained. There is a test for this claim in
  `tests/test_m4.py`.
- **Silent failure.** A deletion request that returns normally and changes nothing. That is what
  we expect past the budget, and this notebook measures how many there are.

## 0. Setup on Colab

Clones and installs the project. Does nothing on a local machine. The guard stops you running
new code against an already-imported old copy, which would look like it worked.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumitasthana/DSR.git"
REVISION = "main"   # Replace with a commit hash to pin this run.

if "src" in sys.modules:
    raise RuntimeError("The experiment package is already imported. "
                       "Restart the runtime before running setup, or you will "
                       "silently test stale code.")

if "google.colab" in sys.modules:
    if not Path("DSR").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    subprocess.run(["git", "-C", "DSR", "fetch", "--all", "--quiet"], check=True)
    subprocess.run(["git", "-C", "DSR", "checkout", REVISION], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "DSR"], check=True)
    os.chdir("DSR")
    print("cloned and installed")
else:
    print("not on Colab; using the existing checkout")

print("working directory:", Path.cwd())

## 1. Load code

Prints which exact version is running. A number from a tree with local changes cannot be
reproduced by anyone else, so the "local changes" line should say none.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import src.common as common
from src.common import load_config, provenance, save_json
from src.run_m4 import run_m4
from src.unlearn import METHODS

REPO = Path(common.__file__).resolve().parents[1]
PROV = provenance()

print("repo          :", REPO)
print("commit        :", PROV["commit"])
print("local changes :", PROV["git_status"] or "none")
print("torch         :", PROV["torch"], "| python", PROV["python"].split()[0])

## 2. Settings

Everything comes from `config.yaml`. The line to notice is the budget: 16, against checkpoints
that go up to 30. Fourteen of those thirty deletions are being asked for after the space has run
out.

In [ ]:
CFG = load_config(REPO / "config.yaml")
M4 = CFG["m4"]
LAMBDAS = CFG["unlearn"]["lambdas"]
SEEDS = CFG["seeds"]
RANK = CFG["projection"]["rank"]
D_IN = CFG["model"]["feature_dim"]
BUDGET = D_IN // RANK
RUN = "nb04-cifar100-full-sweep"

print("resolved m4 block :", json.dumps(M4))
print()
print(f"deletions     : {M4['tasks']} (classes 0 to {M4['tasks'] - 1})")
print(f"read at       : N = {M4['checkpoints']}")
print(f"methods       : {list(METHODS)}")
print(f"lambdas       : {LAMBDAS}")
print(f"seeds         : {SEEDS}")
print(f"budget        : {D_IN} / {RANK} = {BUDGET} deletions")
print(f"past budget   : {M4['tasks'] - BUDGET} of {M4['tasks']} deletions are asked for "
      f"after the space runs out")
print(f"rank sweep    : {M4['rank_sweep']}, predicting budgets "
      f"{[D_IN // r for r in M4['rank_sweep']]}")
print(f"planned work  : {len(SEEDS) * len(METHODS) * M4['tasks']} adapters, "
      f"{len(SEEDS) * len(M4['checkpoints'])} routers, plus the rank sweep")
print("hardware      : CPU only. Roughly fifteen minutes.")

## 3. Connect to storage

Colab discards everything at the end of a session, so results go to object storage. Without
credentials the notebook still runs, it just keeps no copy and says so at the end.

In [ ]:
from src.runstore import connect

OUT = REPO / "outputs" / RUN
OUT.mkdir(parents=True, exist_ok=True)
print("writing results to", OUT)

STORE = connect()
if STORE is None:
    print()
    print("No R2 credentials found. This run will produce results but keep no copy.")
    print("Set DSR_ENDPOINT, DSR_ACCESS_KEY_ID and DSR_SECRET_ACCESS_KEY to keep it.")
else:
    print("R2 bucket    :", STORE.bucket)
    try:
        print("runs on file :", ", ".join(STORE.runs()) or "none yet")
    except Exception as error:
        print("runs on file : could not list,", type(error).__name__)

## 4. Run

The rank sweep goes first, because it is quick and it is the prediction. Then each seed trains
one thirty-deletion sequence per method.

The first run also builds the feature cache, which takes a couple of minutes on top.

In [ ]:
REPORT = run_m4(CFG)

metrics = pd.DataFrame(REPORT["metrics"])
subspace = pd.DataFrame(REPORT["subspace"])
norms = pd.DataFrame(REPORT["fusion_norms"])
sweep = pd.DataFrame(REPORT["rank_sweep"])
per_class = pd.DataFrame(REPORT["per_class"])
print()
print(f"{len(metrics)} metric rows, {len(sweep)} rank-sweep rows, {len(per_class)} class rows")

## 5. Does the limit land where it was predicted?

This is test T2. The budget rule says the space runs out after `64 / rank` deletions, so rank 2
should last 32, rank 4 should last 16 and rank 8 should last 8.

The rule was written down before this ran. If the measured column disagrees with the predicted
column, the rule is wrong.

In [ ]:
rows = []
for rank, part in sweep.groupby("rank"):
    predicted = int(part["predicted_budget"].iloc[0])
    hit_zero = part.loc[part["free_rank_after"] == 0, "task"]
    past = part.loc[part["task"] > predicted, "update_norm"]
    rows.append({"rank": rank, "predicted": predicted,
                 "measured": int(hit_zero.min()) if len(hit_zero) else None,
                 "largest update past it": f"{past.max():.1e}" if len(past) else "not reached"})

sweep_table = pd.DataFrame(rows)
print(sweep_table.to_string(index=False))

T2_HOLDS = bool((sweep_table["predicted"] == sweep_table["measured"]).all())
print()
print("PASS" if T2_HOLDS else "FAIL",
      " the limit lands exactly where 64 / rank says it should")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

for rank, part in sweep.groupby("rank"):
    axes[0].plot([0, *part["task"]], [D_IN, *part["free_rank_after"]],
                 marker="o", markersize=3, label=f"rank {rank}")
axes[0].axhline(0, color="grey", linewidth=0.8)
axes[0].set_xlabel("deletions so far")
axes[0].set_ylabel("directions still free")
axes[0].set_title("The space runs out at 64 / rank")
axes[0].legend(fontsize=8)

one = norms[norms["seed"] == SEEDS[0]]
for strength in [l for l in LAMBDAS if l > 0]:
    part = one[one["lambda"] == strength]
    axes[1].plot(part["task"], part["fused_delta_norm"], label=f"lambda {strength}")
axes[1].axvline(BUDGET, color="red", linestyle="--", linewidth=1, label=f"budget {BUDGET}")
axes[1].set_xlabel("deletions so far")
axes[1].set_ylabel("size of the total change to the head")
axes[1].set_title("The model stops changing at the budget")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Does the model stop changing?

The right-hand plot above should go flat at deletion 16. This checks it as a number rather than
by eye: how much does the total change to the head differ between deletion 16 and deletion 30?

If the answer is zero, every deletion after the sixteenth did nothing at all.

In [ ]:
drift = REPORT["update_drift_past_budget"]
one_seed = norms[(norms["seed"] == SEEDS[0]) & (norms["lambda"] == 1.0)]
at = {int(r.task): r.fused_delta_norm for r in one_seed.itertuples()}
for task in (BUDGET - 1, BUDGET, BUDGET + 1, M4["tasks"]):
    print(f"  after {task:2d} deletions, total change = {at[task]:.6f}")

FROZEN = drift == 0.0
print()
print(f"largest difference between the head at deletion {BUDGET} and at {M4['tasks']}: {drift}")
print("PASS" if FROZEN else "FAIL",
      f" the model is unchanged by every deletion after the {BUDGET}th")

## 7. How many deletion requests did nothing?

The previous cell says the model stops changing. This cell says what that cost.

Each row is one deletion request: the class it was asked to remove, that class's accuracy before
any deletions, and its accuracy at the end. A request that worked drives the number to near zero.
A request that was ignored leaves it high.

The split is at deletion 16, and nothing about these two groups differs except which side of the
budget they fell on.

In [ ]:
summary = per_class.groupby("within_budget")[["accuracy_before", "accuracy_after"]].agg(
    ["count", "mean"]).round(2)
print(summary.to_string())
print()
print(per_class.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
colour = ["tab:green" if w else "tab:red" for w in per_class["within_budget"]]
ax.bar(per_class["task"] - 0.2, per_class["accuracy_before"], width=0.4,
       color="lightgrey", label="before any deletion")
ax.bar(per_class["task"] + 0.2, per_class["accuracy_after"], width=0.4,
       color=colour, label="after all deletions")
ax.axvline(BUDGET + 0.5, color="black", linestyle="--", linewidth=1)
ax.text(BUDGET + 0.7, ax.get_ylim()[1] * 0.9, "budget runs out", fontsize=9)
ax.set_xlabel("deletion request, in the order they were made")
ax.set_ylabel("accuracy on that class %")
ax.set_title("Green: the request worked. Red: the request did nothing.")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8. The Table I comparison

The paper's Table I reports retain and forget accuracy at several values of N. This is the same
shape, from our run.

Read the in-training row down the N column. Its retain accuracy barely moves between N = 20 and
N = 30, which the paper presents as the method being stable at scale. Section 7 above is the
reason: the model is identical in both, because every deletion past the sixteenth was ignored.

In [ ]:
table = (metrics[metrics["lambda"].isin([0.5, 1.0])]
         .groupby(["n", "method", "lambda"])
         [["retain_excluding_forgotten", "forget_all"]].mean().round(2))
print(table.to_string())

print()
print("The paper reports, at N = 30 and lambda 1.0:")
print("  static fusion   retain 12.74  forget  0.77")
print("  MoE router      retain 59.49  forget 59.10")
print("  in-training     retain 58.06  forget 53.83")
print("Absolute numbers cannot match: our backbone starts at 68.83% and theirs at 60.39%.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for method in METHODS:
    part = metrics[(metrics["method"] == method) & (metrics["lambda"] == 1.0)]
    grouped = part.groupby("n")
    axes[0].plot(list(grouped.groups), grouped["retain_excluding_forgotten"].mean(),
                 marker="o", label=method)
    axes[1].plot(list(grouped.groups), grouped["forget_all"].mean(), marker="o", label=method)

for ax, title in zip(axes, ("Accuracy kept on the classes we keep",
                            "Accuracy left on the classes we deleted")):
    ax.axvline(BUDGET, color="red", linestyle="--", linewidth=1, label=f"budget {BUDGET}")
    ax.set_xlabel("number of deletions (N)")
    ax.set_ylabel("accuracy %")
    ax.set_title(title + ", lambda 1.0")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 9. Check against the rules fixed before the run

Four predictions, written into the project's spec before any of this ran.

In [ ]:
D3_DEMONSTRATED = FROZEN and bool((~per_class["within_budget"]).any())
IGNORED = int((~per_class["within_budget"]).sum())
T3_HOLDS = at[BUDGET] == at[M4["tasks"]]

for name, value in [
    (f"T2: the limit is 64 / rank for rank {M4['rank_sweep']}", T2_HOLDS),
    (f"T3: the total change stops growing at deletion {BUDGET}", T3_HOLDS),
    (f"D3: the model is frozen past deletion {BUDGET}", FROZEN),
    (f"D3: deletions past the budget are silently ignored ({IGNORED} of "
     f"{M4['tasks']})", D3_DEMONSTRATED),
]:
    print(f"{'PASS' if value else 'FAIL'}  {name}")

### Plain-language summary

Assembled from the numbers above, not written in advance.

In [ ]:
outside = per_class[~per_class["within_budget"]]
inside = per_class[per_class["within_budget"]]

print(f"We asked the model to delete {M4['tasks']} classes, one at a time.")
print()
print(f"The first {BUDGET} worked. Those classes went from "
      f"{inside['accuracy_before'].mean():.1f}% accuracy to "
      f"{inside['accuracy_after'].mean():.1f}%.")
print()
print(f"The remaining {len(outside)} did nothing. The model after {M4['tasks']} deletions is "
      f"identical to the model")
print(f"after {BUDGET}, to the last decimal place. Those {len(outside)} classes are still at "
      f"{outside['accuracy_after'].mean():.1f}% accuracy,")
print("and the only reason they dropped at all is damage from the earlier deletions.")
print()
print("Nothing reported an error. Each of those requests returned normally.")
print()
print(f"The limit is not a quirk of this setting. Changing the patch size to "
      f"{M4['rank_sweep']} moved the limit")
print(f"to {[D_IN // r for r in M4['rank_sweep']]}, which is exactly 64 divided by the patch "
      f"size in each case.")
print()
print("This matters for reading the paper, which reports results at 20 and 30 deletions and")
print("describes the method as stable there. On our rebuild, stable at that range means frozen.")

## 10. Save and verify

Writes the results, reads them back, and uploads. A run label is written once, so uploading over
an existing one stops rather than overwriting a record something else already cited.

In [ ]:
for name, frame in [("metrics", metrics), ("subspace", subspace), ("fusion_norms", norms),
                    ("rank_sweep", sweep), ("per_class", per_class)]:
    frame.to_csv(OUT / f"{name}.csv", index=False)
table.to_csv(OUT / "table_one_analogue.csv")

save_json(OUT / "metadata.json", {
    "notebook": "notebooks/04_cifar100_full_sweep.ipynb",
    "run_label": RUN, "milestone": "M4",
    "spec_tests": ["T2", "T3"], "spec_discrepancies": ["D2", "D3", "D14", "D15", "D16"],
    "protocol": (f"CIFAR-100, one {M4['tasks']}-deletion sequence per method and seed, read at "
                 f"N = {M4['checkpoints']}, lambda sweep, {len(SEEDS)} seeds, plus a rank sweep "
                 f"over {M4['rank_sweep']}."),
    "interpretation_rule": (
        "T2: free rank reaches 0 at exactly 64/rank for every rank tested. "
        "T3: the fused correction stops growing at the budget. "
        "D3: past the budget the fused head is unchanged and deletion requests are no-ops."
    ),
    "limitations": REPORT["limitations"],
    "results": {
        "clean_accuracy_percent": REPORT["clean_accuracy_percent"],
        "budget": BUDGET,
        "t2_limit_matches_prediction": bool(T2_HOLDS),
        "t3_correction_stops_growing": bool(T3_HOLDS),
        "model_frozen_past_budget": bool(FROZEN),
        "update_drift_past_budget": REPORT["update_drift_past_budget"],
        "deletions_requested": int(M4["tasks"]),
        "deletions_ignored": IGNORED,
        "ignored_class_accuracy_after": float(outside["accuracy_after"].mean()),
        "deleted_class_accuracy_after": float(inside["accuracy_after"].mean()),
    },
    "config": {"m4": M4, "unlearn": CFG["unlearn"], "projection": CFG["projection"],
               "lora": CFG["lora"], "model": CFG["model"]},
    "seeds": SEEDS, "provenance": PROV,
})

for path in sorted(OUT.iterdir()):
    print(f"{path.name:36s} {path.stat().st_size:>9,d} bytes")

assert len(pd.read_csv(OUT / "metrics.csv")) == len(metrics)
assert json.loads((OUT / "metadata.json").read_text(encoding="utf-8"))["run_label"] == RUN
print()
print("reopened and verified")

if STORE is None:
    print()
    print("Not uploaded: no R2 credentials, so this runtime holds the only copy.")
    print("These numbers cannot be cited in a trajectory entry until they are kept.")
else:
    for key in STORE.put(OUT, RUN):
        print("uploaded", key)

## 11. Release the GPU

Not applicable. This notebook never requested a GPU.